In [ ]:
from funciones.cargador import obtener_una_muestra, DATA_DIR
from funciones.detector_entropia import detectar_bursts, plot_muestra, print_diagnostico, plot_espectrograma_3d
import matplotlib.pyplot as plt

import os, glob, json, random
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm
import re
import sys; sys.path.insert(0, r"c:\repos\DroneDetectionRF\NoisyUAV")
from funciones.detector_entropia import detectar_bursts, plot_muestra

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARÁMETROS
# ══════════════════════════════════════════════════════════════════════════════
TARGET  = 4
SNR = 30
INDEX = 0

FS           = 14e6
NPERSEG      = 2048
Z_THRESH     = 3      # más sensible — el filtro de ancho espectral compensa los FA
MIN_BURST_MS = 0.5
MERGE_GAP_MS = 1.0
MIN_Z_ABS    = 4.0
# Discriminador FHSS vs. WiFi:
# P(bin_ruido > BG_MULT × fondo) = 2^(-BG_MULT)  → con BG_MULT=4 → 6.25% → ~128 bins
# FHSS a -12dB añade ~58 bins reales → total ~186
# WiFi OFDM activa >1800 bins de 2048
# Umbral en 25% = 512 bins separa perfectamente FHSS de WiFi
BG_MULT       = 4
MAX_BINS_FRAC = 0.25
SMOOTH_MS = 0.2
ADAPTIVE_WINDOW_MS = 10   # ventana de referencia (ms). 0 = umbral global 


In [ ]:
iq, _, _, _ = obtener_una_muestra(DATA_DIR, target=TARGET, snr=SNR, index=INDEX)

In [ ]:
t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS,
)
print_diagnostico(
    t_ms, nf_v, ns, umbral_v, n_active, bursts,
    nperseg=NPERSEG, fs=FS, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    target=TARGET, snr=SNR, index=INDEX,
)
# ── 1. Paneles de visualización 2D (Reemplaza plt.show()) ───────────────
fig_2d = plot_muestra(
    iq, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
    fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    adaptive_window_ms=ADAPTIVE_WINDOW_MS,
    titulo=f"target={TARGET}  SNR={SNR}dB  index={INDEX}",
)
fig_2d.show()
# ── 2. Visualización 3D Interactiva ─────────────────────────────────────
fig_3d = plot_espectrograma_3d(
    iq,
    fs=FS,
    nperseg=NPERSEG,
    t_lim_ms=None,         # Toma todo el intervalo (downsampling automático activo)
    smooth_sigma=2.0,      # Hace visibles los saltos del burst a bajas SNRs
    floor_pct=5,           # Recorta el suelo de ruido
    titulo=f"target={TARGET}  SNR={SNR}dB  index={INDEX}"
)
fig_3d.show()

# Prueba iterativa

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
#  PARÁMETROS DE EVALUACIÓN
# ══════════════════════════════════════════════════════════════════════════════
OUT_DIR      = r"C:\TFM_data\prueba_entropia"
SAMPLE_FRAC  = 0.30     # fracción de archivos a evaluar por combinación
N_PLOT       = 3        # plots a guardar por combinación (SNR × clase)
NOISE_CLASS  = 4        # target que corresponde a ruido puro (sin dron)
# ── Parámetros del detector ───────────────────────────────────────────────────
FS                 = 14e6
NPERSEG            = 2048
Z_THRESH           = 3.0
MIN_BURST_MS       = 0.5
MERGE_GAP_MS       = 1.0
MIN_Z_ABS          = 4.0
BG_MULT            = 4.0
MAX_BINS_FRAC      = 0.25
SMOOTH_MS          = 0.2
ADAPTIVE_WINDOW_MS = 15.0
# ══════════════════════════════════════════════════════════════════════════════
_PAT = re.compile(r"IQdata_sample(\d+)_target(\d+)_snr([+-]?\d+)\.pt")

In [ ]:
catalog = defaultdict(list)
for f in glob.glob(os.path.join(DATA_DIR, "IQdata_*.pt")):
    m = _PAT.match(Path(f).name)
    if m:
        catalog[(int(m.group(2)), int(m.group(3)))].append(f)

In [ ]:
targets = sorted({k[0] for k in catalog})
snrs    = sorted({k[1] for k in catalog})
print(f"Clases encontradas  : {targets}")
print(f"Niveles de SNR      : {snrs}")
print(f"Total combinaciones : {len(catalog)}")
total_files = int(sum(max(1, round(len(v)*SAMPLE_FRAC)) for v in catalog.values()))
print(f"Archivos a evaluar  : ~{total_files}  "
      f"(≈{SAMPLE_FRAC*100:.0f}% de {sum(len(v) for v in catalog.values())} totales)")
print(f"Plots a generar     : ~{len(catalog)*N_PLOT}")

In [ ]:
class _NpEncoder(json.JSONEncoder):
    """Convierte automáticamente tipos numpy a tipos Python nativos."""
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


def _cargar_iq(path):
    d = torch.load(path, map_location="cpu", weights_only=False)
    return d["x_iq"].float()

def _nombre_clase(target):
    nombres = {0:"DJI_Phantom", 1:"DJI_Mavic", 2:"Parrot_Anafi",
               3:"Hubsan_H501", 4:"Ruido_puro", 5:"Yuneec_Typhoon",
               6:"Syma_X8",    7:"DJI_Inspire", 8:"Cheerson_CX20",
               9:"WLToys_V303", 10:"Walkera_QR"}
    return nombres.get(target, f"target_{target}")

def _detectar(iq):
    """Wrapper que llama al detector con los parámetros globales."""
    return detectar_bursts(
        iq,
        fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
        min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
        min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
        smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS,
    )

PARAMS_DETECTOR = {
    "nperseg": NPERSEG, "z_thresh": Z_THRESH,
    "min_burst_ms": MIN_BURST_MS, "merge_gap_ms": MERGE_GAP_MS,
    "min_z_abs": MIN_Z_ABS, "bg_mult": BG_MULT,
    "max_bins_frac": MAX_BINS_FRAC, "smooth_ms": SMOOTH_MS,
    "adaptive_window_ms": ADAPTIVE_WINDOW_MS,
}

os.makedirs(OUT_DIR, exist_ok=True)
summary_global = {}

for target in tqdm(targets, desc="Clases"):
    clase_dir = os.path.join(OUT_DIR, _nombre_clase(target))
    os.makedirs(clase_dir, exist_ok=True)
    resumen_clase = {}

    for snr in tqdm(snrs, desc=f"  SNR [{_nombre_clase(target)}]", leave=False):
        files = catalog.get((target, snr), [])
        if not files:
            continue

        n_sample = max(1, round(len(files) * SAMPLE_FRAC))
        sample   = random.sample(files, n_sample)
        snr_dir  = os.path.join(clase_dir, f"SNR_{snr:+d}dB")
        os.makedirs(snr_dir, exist_ok=True)

        resultados_snr       = []
        seleccionados_plot   = []   # (path, iq, resultado_detección)

        for path in sample:
            iq = _cargar_iq(path)
            t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = _detectar(iq)

            nombre = Path(path).stem
            r = {
                "archivo":     nombre,
                "n_bursts":    len(bursts),
                "detectado":   len(bursts) > 0,
                "noise_floor": round(float(np.median(nf_v)), 4),
                "noise_sigma": round(float(ns), 5),
                "umbral":      round(float(np.median(umbral_v)), 4),
                "bursts": [
                    {"t0":      round(b["t0"],     2),
                     "t1":      round(b["t1"],     2),
                     "dur_ms":  round(b["dur_ms"], 2),
                     "drop_b":  round(b["drop_b"], 3),
                     "z_peak":  round(b["z_peak"], 1),
                     "n_act":   int(b["n_act"])}
                    for b in bursts
                ]
            }
            resultados_snr.append(r)

            if len(bursts) > 0:
                seleccionados_plot.append(
                    (path, iq, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts))

        # Si ninguno tiene bursts, plotear los N_PLOT primeros de todos modos
        if not seleccionados_plot:
            for path in sample[:N_PLOT]:
                iq = _cargar_iq(path)
                t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = _detectar(iq)
                seleccionados_plot.append(
                    (path, iq, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts))

        # Guardar N_PLOT plots
        random.shuffle(seleccionados_plot)
        for idx, (path, iq, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts) \
                in enumerate(seleccionados_plot[:N_PLOT]):
            nombre = Path(path).stem
            titulo = f"{_nombre_clase(target)}  SNR={snr:+d}dB  [{nombre}]"
            try:
                fig = plot_muestra(
                    iq, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
                    fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
                    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
                    adaptive_window_ms=ADAPTIVE_WINDOW_MS,
                    titulo=titulo,
                )
                fig.savefig(os.path.join(snr_dir, f"plot_{idx+1:02d}_{nombre}.png"),
                            dpi=100, bbox_inches='tight')
                plt.close(fig)
            except Exception as e:
                print(f"    [WARN] plot fallido: {nombre}: {e}")

        # Estadísticas del SNR
        n_total      = len(resultados_snr)
        n_detectados = sum(1 for r in resultados_snr if r["detectado"])
        es_dron      = (target != NOISE_CLASS)
        Pd  = n_detectados / n_total if es_dron     else float("nan")
        Pfa = n_detectados / n_total if not es_dron else float("nan")

        resumen_snr = {
            "snr":          snr,
            "n_total":      n_total,
            "n_detectados": n_detectados,
            "Pd":           round(Pd,  4) if not np.isnan(Pd)  else None,
            "Pfa":          round(Pfa, 4) if not np.isnan(Pfa) else None,
            "archivos":     resultados_snr,
        }
        resumen_clase[snr] = resumen_snr

        with open(os.path.join(snr_dir, "estadisticas.json"), "w", encoding="utf-8") as jf:
            json.dump(resumen_snr, jf, indent=2, ensure_ascii=False, cls=_NpEncoder)


    # JSON resumen de la clase
    with open(os.path.join(clase_dir, "resumen_clase.json"), "w", encoding="utf-8") as jf:
        json.dump({"clase": _nombre_clase(target), "target": target,
                "parametros": PARAMS_DETECTOR,
                "por_snr": resumen_clase},
                jf, indent=2, ensure_ascii=False, cls=_NpEncoder)

    # Curva Pd/Pfa vs SNR
    snr_vals = [s for s in snrs if s in resumen_clase]
    Pd_vals  = [resumen_clase[s]["Pd"]  for s in snr_vals]
    Pfa_vals = [resumen_clase[s]["Pfa"] for s in snr_vals]

    plt.style.use('dark_background')
    fig_c, ax_c = plt.subplots(figsize=(12, 5))
    fig_c.patch.set_facecolor('#1a1a2e'); ax_c.set_facecolor('#16213e')
    ax_c.tick_params(colors='#e0e0e0')
    for sp in ax_c.spines.values(): sp.set_color('#444')

    if target != NOISE_CLASS:
        ax_c.plot(snr_vals, Pd_vals,  'o-',  color='#3498db', lw=2, ms=6, label='Pd')
    else:
        ax_c.plot(snr_vals, Pfa_vals, 's--', color='#e74c3c', lw=2, ms=6, label='Pfa')
    ax_c.axvline(-12, color='#9b59b6', ls='--', lw=1.5, alpha=0.7, label='SOTA −12dB')
    ax_c.axhline(0.9,  color='#3498db', ls=':', lw=1, alpha=0.4)
    ax_c.axhline(0.1,  color='#e74c3c', ls=':', lw=1, alpha=0.4)
    ax_c.set_xlabel('SNR (dB)', color='white')
    ax_c.set_ylabel('Probabilidad', color='white')
    ax_c.set_ylim(-0.05, 1.05)
    ax_c.set_title(
        f'{_nombre_clase(target)}  —  Pd/Pfa vs SNR  '
        f'(z={Z_THRESH}  minBurst={MIN_BURST_MS}ms  minZ={MIN_Z_ABS}  '
        f'bgMult={BG_MULT}  maxBins={MAX_BINS_FRAC*100:.0f}%)',
        color='white')
    ax_c.legend(facecolor='#1a1a2e', labelcolor='white')
    ax_c.grid(color='#2c3e6b', alpha=0.3)
    plt.tight_layout()
    fig_c.savefig(os.path.join(clase_dir, "pd_pfa_vs_snr.png"), dpi=130, bbox_inches='tight')
    plt.close(fig_c)

    summary_global[_nombre_clase(target)] = resumen_clase

print(f"\n✓ Todo guardado en: {OUT_DIR}")
print(f"  Estructura: clase / SNR_Xdb / {{plot_01..0{N_PLOT}.png, estadisticas.json}}")
print(f"              clase / {{resumen_clase.json, pd_pfa_vs_snr.png}}")
